# AgentLens 1 — Data Collection

How the *(prompt, tool)* dataset is collected and labeled. The agent/LLM, SSH,
and the Linux VMs are deliberately **out of scope here** — this notebook shows
the collection mechanism and the resulting labeled dataset, nothing else.

**Output dataset:** `cyber_logs.csv` — one row per query, with the tool the LLM
chose and the correct tool.

## 1. The collection mechanism: a logging wrapper

Every tool is wrapped so that calling it records a `(prompt, tool)` pair before
it runs. That shared list *is* the trajectory data. Tiny self-contained demo
(no network, no LLM):

In [1]:
logs = []

def logging_tool(tool_name, func):
    "Wrap a tool so each call appends (prompt, tool) to the shared log."
    def wrapped(prompt):
        logs.append({"prompt": prompt, "tool": tool_name})
        return func(prompt)
    return wrapped

# three illustrative tools (no real work)
nmap   = logging_tool("NmapScan",      lambda p: "ran nmap")
auth   = logging_tool("ReadAuthLog",   lambda p: "ran tail auth.log")
block  = logging_tool("BlockIP",       lambda p: "ran iptables deny")

# the agent calls whichever tool a task needs; every call is logged
nmap("scan the target for open services")
auth("show the recent authentication log")
block("block the suspicious ip address")

import pandas as pd
pd.DataFrame(logs)

,prompt,tool
0,scan the target for open services,NmapScan
1,show the recent authentication log,ReadAuthLog
2,block the suspicious ip address,BlockIP


## 2. The full collected dataset

Running the agent over 640 labeled queries produced `cyber_logs.csv`.
Columns:
- `prompt` — the task in natural language
- `tool_predicted` — the tool the **LLM** chose (the baseline)
- `tool_ground_truth` — the **correct** tool (the label the classifier learns)
- `difficulty` — `easy` / `hard`;  `category` — query style

In [2]:
import os
import pandas as pd

# find the dataset whether this notebook sits beside the CSV or in the repo
for p in ["cyber_logs.csv",
          os.path.join("..", "data", "trajectories", "cyber_logs.csv"),
          os.path.join("data", "trajectories", "cyber_logs.csv")]:
    if os.path.exists(p):
        CSV = p; break
else:
    raise FileNotFoundError("cyber_logs.csv not found -- put it next to this notebook")
df = pd.read_csv(CSV)
print("Loaded", CSV, "->", df.shape[0], "rows")
df[["prompt", "tool_predicted", "tool_ground_truth", "difficulty", "category"]].head(8)

Loaded cyber_logs.csv -> 640 rows


,prompt,tool_predicted,tool_ground_truth,difficulty,category
0,Do a port scan of the defender that also repor...,PortScan,NmapScan,hard,opposite
1,Run nmap -p 1-1000 against 192.168.56.102,NmapScan,PortScan,easy,direct
2,Scan ports 1-1000 on the defender host,NmapScan,PortScan,easy,direct
3,Is Apache 2.4.49 vulnerable to any known explo...,CheckVulnerability,CheckVulnerability,easy,direct
4,Is ssh 1.18 vulnerable to any known exploits?,CheckVulnerability,CheckVulnerability,easy,direct
5,Run a scan against the target VM,NmapScan,NmapScan,hard,ambiguous
6,"Once we know it's nginx 8.0.32, check it for k...",CheckVulnerability,CheckVulnerability,hard,multistep
7,Can you get me onto the defender host?,SSHConnect,SSHConnect,hard,natural


## 3. Label distribution

In [3]:
print("Tools (classes):", df["tool_ground_truth"].nunique())
print(df["tool_ground_truth"].value_counts().to_string())
print("\nDifficulty:", dict(df["difficulty"].value_counts()))
print("Category:  ", dict(df["category"].value_counts()))

Tools (classes): 11
tool_ground_truth
CheckVulnerability    81
PortScan              68
NmapScan              67
ListeningPorts        60
ReadSyslog            57
ListProcesses         55
BlockIP               55
ReadAuthLog           54
CheckFailedLogins     49
GetSystemInfo         49
SSHConnect            45

Difficulty: {'easy': 440, 'hard': 200}
Category:   {'direct': 440, 'opposite': 40, 'ambiguous': 40, 'multistep': 40, 'natural': 40, 'trick': 40}


## 4. LLM accuracy on the collected data (baseline + Table I)

How often the LLM picked the correct tool — overall, on hard queries, and by
category. This is the baseline the trained classifier is compared against.

In [4]:
df["llm_correct"] = df["tool_predicted"] == df["tool_ground_truth"]
print(f"LLM overall : {df['llm_correct'].mean():.1%}  ({df['llm_correct'].sum()}/{len(df)})")
print(f"LLM on hard : {df.loc[df['difficulty']=='hard','llm_correct'].mean():.1%}")

order = ["direct", "ambiguous", "multistep", "trick", "natural", "opposite"]
tab1 = (df.groupby("category")["llm_correct"].agg(n="size", llm_accuracy="mean")
          .reindex([c for c in order if c in df["category"].unique()]))
tab1["llm_accuracy"] = (tab1["llm_accuracy"]*100).round(1).astype(str) + "%"
tab1

LLM overall : 79.4%  (508/640)
LLM on hard : 73.5%


,n,llm_accuracy
category,,
direct,440,82.0%
ambiguous,40,57.5%
multistep,40,70.0%
trick,40,72.5%
natural,40,82.5%
opposite,40,85.0%


**Takeaway.** 640 labeled examples, 11 tools, balanced classes. The LLM gets
~79% of tool choices right (lower on hard/ambiguous queries) — leaving clear room
for a trained classifier, which is Notebook 2.